# Information about the dataset

- Dataset source: the dataset 'Extrovert vs. Introvert Behavior Data' is available on Kaggle, at https://www.kaggle.com/datasets/rakeshkapilavai/extrovert-vs-introvert-behavior-data?resource=download - Dataset Source:The dataset is available on Kaggle: MxMH Survey Results, at https://www.kaggle.com/datasets/catherinerasgaitis/mxmh-survey-results

- Structure: It contains 2900 rows and 8 columns, including 5 numerical features, 2 binary features a 1 categorical (target).

- Missing values: The dataset contains 458 missing values

- Features:   

    - Time_spent_Alone: Hours spent alone daily (0–11).
    - Stage_fear: Presence of stage fright (Yes/No).
    - Social_event_attendance: Frequency of social events (0–10).
    - Going_outside: Frequency of going outside (0–7).
    - Drained_after_socializing: Feeling drained after socializing (Yes/No).
    - Friends_circle_size: Number of close friends (0–15).
    - Post_frequency: Social media post frequency (0–10).
    - Personality: Target variable (Extrovert/Introvert).

# Aim of the project

The goal is to develop a binary classifier that predicts whether a person is Extrovert (1) or Introvert (0) from social-behavior features (hours alone, event attendance, going out, friends size, posting frequency, stage fear, and whether they feel drained after socializing). 

# Import

In [ ]:
import pandas as pd
import numpy as np
!pip install missingno
!pip install imblearn
import missingno as msno 
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer 
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline as IMBPipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import TomekLinks
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate, RepeatedStratifiedKFold, learning_curve, validation_curve
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report, accuracy_score
from imblearn.pipeline import Pipeline as IMBPipeline
from scipy.stats import loguniform
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, ConfusionMatrixDisplay, auc

Use the Pandas method read_csv to import the dataset.

In [ ]:
df = pd.read_csv("personality_dataset.csv")

# Data Visualization

Retrieve the dataset's dimensions using the shape attribute.

In [ ]:
print(df.shape) 

Features: 8
Elements: 2900 

<hr>

Display the dataset

In [ ]:
df

Obtain details about the dataset's features using the info() method. It confirms number of rows and non-null counts per column and shows each column's dtype.

In [ ]:
df.info()

# Dealing with missing values

Calculate the number and the percentage of missing values per column.

In [ ]:
missing_counts = df.isna().sum(axis=0)
print("Missing counts per column:\n", missing_counts)

missing_percentage = missing_counts / df.shape[0] * 100
print("\nPercentage of missing entries per column:\n", missing_percentage)

Sort by highest fraction of missing values, to see which columns have the largest percentage of missing values

In [ ]:
missing_percentage.sort_values(ascending=False)

Using the missingno.matrix method I can visualize in the feature matrix the distribution of missing values in the dataset 

In [ ]:
msno.matrix(df)

In this dataset, the proportion of missing values is minimal and unlikely to significantly affect analysis. Indeed, every column has under 3% of its entries missing, it's usually better to impute those missing values rather than drop an entire column.

<hr>

Let's visualize the distributions of the features: draw a histogram of the five numeric features. The histogram is the most direct way to see the data distribution before any modeling: they are useful for understanding at a glance whether the values ​​are concentrated in one area or spread across the entire range, whether the distribution is unbalanced, whether there are long tails or anomalous peaks that may suggest outliers. These indications guide concrete preprocessing choices.

In [ ]:
#Identify the numeric columns
numeric_cols = [
    "Time_spent_Alone",
    "Social_event_attendance",
    "Going_outside",
    "Friends_circle_size",
    "Post_frequency"
]
# Plot histogram for each numeric feature
plt.figure(figsize=(12, 8))
for i, col in enumerate(numeric_cols, start=1):
    plt.subplot(2, 3, i)
    sns.histplot(df[col], bins=12, kde=False, edgecolor="k")
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
plt.tight_layout()
plt.show()

Looking at the histograms, the features appear to be discrete/ordinal numeric, with different scales, slight asimmetries and no obvious outliers. Consequently, the few missing values ​​of the numeric variables will be imputed with the median,because it is robust to the observed tails.
It is also useful to bring all the variables to the same scale with StandardScaler(), to prevent those with a wider range from having a greater weight.

Let's plot a correlation heatmap

In [ ]:
#Compute correlation matrix 
df_numeric = df[numeric_cols].dropna()
corr_mat = df_numeric.corr()

# Plot the correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_mat,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    cbar=True,
    linewidths=0.5
)
plt.title("Correlation Heatmap (numeric features)")
plt.show()

The heatmap shows very strong relationships among the five numeric features: Time_spent_Alone is highly negatively correlated with Social_event_attendance , Going_outside , Friends_circle_size , Post_frequency. While the other four features are all highly positively intercorrrelated.

# Data Transformation Pipeline

The data transformation pipeline specifies which transformation should be applied to each feature. In this case, the transformation pipeline is the following: 

<img src="image-20250825-171737.png" width="" align="" />

For the numerical features Time_spent_Alone , Social_event_attendance , Going_outside , Friends_circle_size and Post_frequency the pipeline is composed by two transformers: 

1. SimpleImputer(strategy="median") : each column has <3% of missing values, then fills the few NaNs robustly;

2. StandardScaler: brings all the numbers on the same scale

In [ ]:
numerical_columns = [
    "Time_spent_Alone", 
    "Social_event_attendance",
    "Going_outside",
    "Friends_circle_size",
    "Post_frequency"
]

pipeline_numerical = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

While for the categorical (binary) features Stage_fear and Drained_after_socializing , let's apply the transformers:

1. SimpleImputer(strategy="most_frequent"): since both features are categorical, it fills the few NaN with the mode.

2. OrdinalEncoder: it maps No-->0, Yes-->1 to obtain numerical variables

In [ ]:
binary_columns = ["Stage_fear" , "Drained_after_socializing"]

pipeline_binary = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal_encoder", OrdinalEncoder(categories=[["No", "Yes"], ["No", "Yes"]]))
])

### Definition of the final ColumnTransformer

Let's use the ColumnTransformer to combine the two previously defined pipelines into a single transformer: it routes each column group to its own pipeline and produces a single numeric matrix ready for the model, ensuring a coherent and streamlined transformation process. Unlisted columns are discarded.

In [ ]:
data_transformer = ColumnTransformer(
    transformers=[
        ("numerical", pipeline_numerical, numerical_columns),
        ("binary", pipeline_binary, binary_columns),
    ],
    remainder="drop",
    verbose_feature_names_out=False 
        )

### Target mapping

Let's map labels in 0/1 for the binary classification

In [ ]:
X = df[numerical_columns + binary_columns].copy() # Features
Y = df["Personality"].map({"Extrovert": 1, "Introvert": 0}) # Target

# Training and Test Sets

Split the entire dataset into training and test sets, putting 20% of the instances in the test set (test_size=0.2) and the remaining 80% in the training set, through the train_test_split() method: it ensures the test set represents unseen data, enabling a reliable evaluation of the model's performance. Then, apply stratified sampling to maintain the same extrovert/introvert ratio in both the training and testing sets, preventing imbalances that could negatively affect model training and evaluation. random_state=42 makes the split reproducible, meaning that I'll obtain the same split if I run the code multiple times.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y,  
    test_size=0.20,
    stratify=Y,  
    random_state=42
)

### Data Transformation

Fit the data transformer only on X_train: this means that the transformer learns its parameters using only training data. These parameters are the medians for imputing NaNs in the 5 numerics; mean and standard deviation for the StandardScaler for the numerics; mode for imputing NaNs in the binary values; and categories for the OrdinalEncoder. Then, apply the same transformation to both the train and test data, in order to avoid using information from the test set to choose mean/median/deviation, that would lead to data leakage.

In [ ]:
data_transformer.fit(X_train) # Fit the ColumnTransformer on the training set

X_train_tr = data_transformer.transform(X_train) # Transform the training set
X_test_tr  = data_transformer.transform(X_test) # Transform the test set

For readability, a DataFrame with the names of the transformed features is created in order to verify that transformations applied as intended and missing values have been correctly imputed.

In [ ]:
feat_names = data_transformer.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_tr, columns=feat_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_tr, columns=feat_names , index=X_test.index)

# Verify the transformations
print("Shape of X_train after transformation:", X_train_df.shape)
print("Shape of X_test after transformation:", X_test_df.shape)

# Preview the transformed datasets
print("Preview of transformed training set:")
print(X_train_df.head())
print("\nPreview of transformed test set:")
print(X_test_df.head())

The shape is now (2320, 7) for the train and (580, 7) for the test: exact for an 80/20 split of 2900 rows and 7 output features.

Now let's check that each feature in the training set and test set does not contain NaNs. Each row in the output lists a feature with the number of NaNs: if they are all 0, the imputation correctly was perfomed correctly.

In [ ]:
# Check missing values in the transformed training set
print("Missing values in the transformed training set:")
print(X_train_df.isnull().sum())

# Check missing values in the transformed test set
print("Missing values in the transformed test set:")
print(X_test_df.isnull().sum())

# Model Selection

Model selection is the process of identifying the most suitable model family for the problem, choosing its hyperparameters, and estimating its ability to generalize to unseen data. To obtain a reliable estimate, multiple alternatives must be compared: in order to achieve this, we need a pipeline that includes
- Transformation: transforms data with ColumnTransformer (imputation, encoding, scaling) to avoid any leakage.
- Sampler: handles imbalanced classes, in order to let the model learn equally from each class.
- Classifier: applies a machine learning algorithm to predict the target variable based on the processed data. In this case a Random Forest is appropriate for the following reasons:
    - it handles nonlinearities and collinearities , 
    - it works with mixed numeric/binary features,
    - it is robust to small datasets.

Dimensionality reduction is not applied because the dataset only has 7 features and it is preferable to prioritize interpretability.

This pipeline is a baseline, the choice of the final classifier will take place in the model selection section.

In [ ]:
model_pipeline = IMBPipeline([
    ('transformation', data_transformer),
    ('sampler', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2,
        class_weight="balanced", random_state=42))
])

For sanity check, run the fit and the predict method: before running cross-validation and tuning, verify that all pipeline steps are connected correctly. These predicitions are not used to assess model performance

In [ ]:
model_pipeline.fit(X_train,y_train)

In [ ]:
model_pipeline.predict(X_test)

This output is the list of predicted classes for each row of X_test: each element is the class predicted by the model for that row.

## Configurations

For each step of the pipeline model_pipeline, define a list of admissible parameters: the idea is to identify a group of feasible choices; and then perform the Cartesian product between these groups to generate all the configurations to test.

### Sampler's configuration


- passthrough (“no sampler”): does not apply any balancing. It's used when the classes are already balanced or in order to avoid introducing bias/overfitting with oversampling.
- SMOTE (Synthetic Minority Oversampling Technique): improves class balance by generating synthetic samples for the minority class. It does this by interpolating between each minority point and its k closest neighbors (it doesn't simply duplicate existing ones). It is used to increase minority presence and smooth the decision boundary.
- RandomOverSampler: balances the dataset by randomly duplicating existing samples from the minority class, increasing its representation, until the desired ratio is achieved.
- TomekLinks: identifies pairs of samples from different classes that are very close to each other and removes the majority point , effectively cleaning the decision boundary and reducing noise. Mostly used with noisy data and fuzzy boundaries.

In [ ]:
sampler_configs = [
    {"sampler": ["passthrough"]},  

    {
        "sampler": [SMOTE(random_state=42)],     
        "sampler__sampling_strategy": [1.0],   # final ratio we want the classes to reach after oversampling.
    },

    {
        "sampler": [RandomOverSampler(random_state=42)],   
        "sampler__sampling_strategy": [1.0],
    },

    {
        "sampler": [TomekLinks()],  
        "sampler__sampling_strategy": ["not minority"],  # remove only examples of the majority 
    }    
]

### Classifiers' configuration


- RandomForestClassifier : ensemble learning algorithm that combines multiple decision trees, each trained on bootstrapped (random, drawn with replacement) subsets of the training data. 
- LogisticRegression: linear model that predicts class probabilities based on the logistic (sigmoid) function. 
- SVC: supervised learning algorithm that identifies the hyperplane that maximizes the margin between data points of different classes. 
- KNeighborsClassifier (KNN): instance-based classifier that predicts the class of a data point by considering its k-nearest neighbors in the feature space and voting among the neighbors.


In [ ]:
classifier_configs = [
    {
        "classifier": [RandomForestClassifier(random_state=42)],
        "classifier__n_estimators": [200, 400],  # Number of trees to try 
        "classifier__max_depth": [None, 15],  # Max depth of each tree
        "classifier__min_samples_leaf": [1, 3],  # Minimum number of samples required to be at a leaf node
        "classifier__max_features": ["sqrt", 0.8],  # Number of features to consider at each split
        "classifier__class_weight": ["balanced", None],  # To handle class imbalance
    },

    {
        "classifier": [LogisticRegression(max_iter=1000)],
        "classifier__penalty": ["l2"],  # L2 is the regularization type (standard and robust)
        "classifier__C": [0.5, 1, 3],  # Strenght of regularization
        "classifier__solver": ["liblinear", "lbfgs"],  # Optimization algortihm
        "classifier__class_weight": ["balanced", None],
    },

    {
        "classifier": [SVC(random_state=42)],   
        "classifier__kernel": ["linear", "rbf"],  #  two kernel: linear and not linear (rbf)
        "classifier__C": [0.5, 1, 3],  
        "classifier__gamma": ["scale"],         # Controls rbf kernel's curvature
        "classifier__class_weight": ["balanced", None],
    },

    {
        "classifier": [KNeighborsClassifier()],
        "classifier__n_neighbors": [5, 9],  # Number of neighbors to consider
        "classifier__weights": ["uniform", "distance"],  # How to weight neighbors: each one with equal weight or closer neighbors have more weight
        "classifier__p": [1, 2],                # 1=Manhattan, 2=Euclidean
    },
]


Generate all possible configurations for the pipeline by combining the sampler and classifier configurations using itertools.product. This process creates a list of all parameter combinations that can be tested during model selection.

In [ ]:
all_configs = [dict(
    itertools.chain(*(e.items() for e in configuration))) 
    for configuration in itertools.product(
        sampler_configs,  
        classifier_configs)]

In [ ]:
f'Number of all possible configurations: {len(all_configs)}'

## Model Selection with Nested Cross-Validation

Nested cross-validation is useful to find the best configuration among different machine learning algorithms to avoid leakage. It is composed by:
 - Inner loop: use RandomizedSearchCV for hyperparameter optimization (to choose the best combination within each training fold)
 - Outer loop: use cross_validate with the best candidate alrealdy identified in the inner loop as hyperparameter.

The inner loop chooses the best model and hyperparameters using only the training data set in that outer fold. Let's use RandomizedSearchCV: it randomly extracts a certain number of combinations (sampler + classifier + hyperparameters) from our search space and evaluates them with an internal cross-validation (in this case, 2-fold cross-validation).
For each of the chosen combinations, 2-fold cross-validation divides the inner training into two stratified parts: one half is trained (preprocessing --> sampler --> model) and the other is evaluated, then the roles are reversed. 
Use F1 as the metric score because it balances precision and recall on the positive class (in this case, Extrovert = 1). Therefore, the two F1s of each combination are averaged: 
the combination with the best average F1 is chosen, and the entire pipeline is retrained from scratch using it.

In [ ]:
# Inner loop
inner_search = RandomizedSearchCV(
    estimator=model_pipeline,            # pipeline to optimize: transformation → sampler → classifier
    param_distributions=all_configs,     # configurations to test
    n_iter=len(all_configs) * 5,         # explore 16*5=80 of the total configurations 
    n_jobs=-1,                           # number of jobs to run in parallel
    cv=2,                                # 2-fold cross-validation
    scoring="f1",                        # scoring metric
    error_score="raise"                  # display errors
)

In short, inner_search takes the pipeline as an estimator and randomly tries n_iter combinations from the all_configs dictionary; each combination is evaluated with cv=2 using F1; everything runs in parallel (n_jobs=-1). Finally, for the current outer fold, inner_search knows the best combination found on that subtraining.

The outer loop measures the quality of the model already optimized by the inner loop. Implement the outer loop using the cross_validate function to evaluate model performance. Use F1 as the scoring metric and perform 5-fold cross-validation to ensure robust evaluation. Set return_estimator=True  to retrieve the best estimators for each fold, enabling detailed analysis of the selected models.

In [ ]:
scores = cross_validate(
    estimator = inner_search,                    # RandomizedSearchCV object
    X = X_train,
    y = y_train,
    scoring='f1',                                # Scoring metric
    cv = 5,                                      # 5-fold cross-validation
    return_estimator = True,                     # Return estimators for further analysis
    verbose = 3,
    error_score =  'raise'
)

cross_validate divides X_train, y_train into 5 outer folds. For each outer fold:
- it uses the training data from that fold to start the inner loop (inner_search) with the internal 2-fold CV;
- it takes the best combination found inside;
- it evaluates that combination against the validation of the outer fold and saves the F1 in scores['test_score'],
- it also saves the best model object in scores['estimator'] thanks to return_estimator=True.
Finally, I'll get a dictionary with the 5 F1 scores (one per fold) and the corresponding 5 selected models, which are the best pipelines found in the inner loop.

It's useful to inspect each fold to see which combination (sampler, classifier, hyperparameters) was selected in each split and with what F1 it performed in validating that fold, before choosing a final 'winning' pipeline.

In [ ]:
for index, estimator in enumerate(scores['estimator']):
    print(f"Fold {index + 1}:")
    print(f"  Sampler: {estimator.best_estimator_.get_params().get('sampler', None)}")
    print(f"  Classifier: {estimator.best_estimator_.get_params().get('classifier', None)}")
    if estimator.best_estimator_.get_params().get('classifier', None):
        print(f"    Classifier Parameters: {estimator.best_estimator_.get_params()['classifier'].get_params()}")
    print(f"  F1 Score: {scores['test_score'][index]}")
    print('-'*40)

Looking at the output, the performance of the models varies depending on the configurations of samplers and classifiers. 
- In Fold 1, no sampler is applied, and the best classifier is a SVC This configuration achieves an F1 score of 0.947.
- In Fold 2, RandomOverSampler is applied, and the best classifier is a SVC. The performance is the highest among the folds, with F1 = 0.954.
- In Fold 3, TomekLinks is used to clean the decision boundaries by removing ambiguous pairs, and a RandomForestClassifier is used as the classifier. The F1 score is 0.927.
- In Fold 4, the sampler used is again TomekLinks and RandomForestClassifier is the best model: this configuration achieves an F1 = 0.947.
- In Fold 5, TomekLinks is applied, while the best classifier is SVC. The F1 score is 0.914.

Now, take those 5 best candidates (one per fold) and train them from scratch on the entire training set which was set aside at the beginning (X_train, y_train). Measure their F1 on training (to understand how well the model can "explain" the data it has seen) and the F1 on testing (X_test, y_test, completely unseen).

In [ ]:
for estimator in scores['estimator']:
    pred_train = estimator.best_estimator_.fit(X_train, y_train)  # Train the candidate model of that fold from scratch on the entire training set.
    pred_train = estimator.best_estimator_.predict(X_train)  # Get the predictions on the training set with the newly retrained model
    pred_test = estimator.best_estimator_.predict(X_test)   # Predictions also on the test set kept aside 
    f1_train = f1_score(y_train, pred_train)   # Calculate the F1 on the training
    f1_test = f1_score(y_test, pred_test)    # Calculate the F1 on the test set
    print(f'F1 on training set:{f1_train}, F1 on test set:{f1_test}')

The training F1 settles around 0.94 for all models, while the test F1 is around 0.917-0.925. This uniformity is a sign that the solutions identified by the nested CV are stable: even starting from different configurations, once the same data transformation is applied and the hyperparameters within the folds are optimized, the models tend to make the same decisions on the examples in the test set.
The gap between training and testing is small and consistent with what was seen in cross-validation: it indicates that we are not over-fitting the training set and that the pipeline generalizes well even to previously unseen data.

Before moving on to a more refined search for the hyperparameters, it is useful to fix only one candidate among the 5. The most appropriate criterion is to choose the model that achieved the best F1 on the external validation folds.

In [ ]:
best_idx = int(np.argmax(scores['test_score']))   # Find the index of the fold with the highest F1
best_score = float(scores['test_score'][best_idx])   # Get the best F1 score 

best_search_cv = scores['estimator'][best_idx]   # Get the RandomisedSearchCV object for the best fold
candidate = best_search_cv.best_estimator_   # Get the best pipeline from the best fold

print(f"Best model: {best_idx + 1}")
print(f"F1 on the best model: {scores['test_score'][best_idx]:.3f}")
candidate   # Visualize the pipeline

The second fold performed better on its validation set.
The ColumnTransformer inputes (with SimpleImputer) and scales (with StandardScaler) numerics, while inputes (with SimpleImputer) and encodes (with OrdinalEncoder) binary. Once the transformation is complete, the pipeline moves on to the sampler: the RandomOverSampler, which duplicates examples from the minority class during the fit to balance the fold's training set. The last block is the classifier, here an SVC: it receives the already imputed, scaled/encoded, and rebalanced dataset as input and learns the decision boundary. 

## Refinement of the selected model 

Now, tune the hyperparameters of the candidate model for optimal performance.  Define the pipeline of the selected model:

In [ ]:
best_model_pipeline = IMBPipeline([
    ('transformation', data_transformer),    
    ('sampler', RandomOverSampler(random_state=42, sampling_strategy=1.0)),
    ('classifier', SVC(kernel='rbf', C=1.0, gamma='scale', class_weight=None, random_state=42))
])

The list of possible hyperparameters for the best model pipeline includes SMOTE’s sampling_strategy, and SVC’s kernel (rbf/linear), C, gamma (for RBF), and class_weight.

In [ ]:
params = {
    'sampler__sampling_strategy': [1.0],   # Oversampling of the minority class
    'classifier__kernel': ['rbf', 'linear'],   # SVC: Let's try both linear and RBF margin.    
    'classifier__C': loguniform(1e-2, 1e2),  # C controls how much the model accepts training errors
    'classifier__gamma': loguniform(1e-3, 1e0),   # Gamma controls the “curvature” of the RBF kernel
    'classifier__class_weight': [None, 'balanced']
}

Define a RandomizedSearchCV object to identify the best configuration for the above model pipeline. RandomizedSearchCV randomly extracts 30 combinations from the params values ​​and, for each, estimate the F1 using a stratified 5-fold cross-validation. The combination with the highest average F1 is chosen as the best. Then, run the fitting method.

In [ ]:
rs_best = RandomizedSearchCV(
    estimator=best_model_pipeline,      # pipeline: transformation → RandomOverSampler → SVC
    param_distributions=params,         # hyperparameters space
    n_iter=30,                          # number of combinations to try  
    scoring='f1',                       # f1 metrics
    cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=42),                                                               
    random_state=42,                    
    error_score='raise'                 
)

In [ ]:
rs_best.fit(X_train, y_train)    # runs CV and selects the config with the highest average F1

Finally, inspect the selected model: respectively, represent the final pipeline (rs_best.best_estimator_) and the chosen values (rs_best.best_params_)

In [ ]:
rs_best.best_estimator_

In [ ]:
rs_best.best_params_

In [ ]:
f1_score(y_test, rs_best.best_estimator_.predict(X_test))

In the refinement phase, the candidate model architecture was kept unchanged: the pipeline remained composed of the same data transformations (imputation and scaling/encoding via a ColumnTransformer), the same minority-class rebalancing with RandomOverSampler, and the same SVC classifier. The aim of the refinement was to identify hyperparameter values that improve generalization. For the oversampling part, the RandomOverSampler’s sampling_strategy was tuned. For the SVC, the kernel type (linear vs. RBF), the penalty parameter C, the RBF kernel parameter gamma, and the class_weight option (none vs. balanced) were varied. The search selected the combination that maximized the average F1 across the cross-validation folds.

# Evaluation

### F1, precision, recall and accuracy

The evaluation aims to honestly measure the pipeline's performance on unseen test data, providing a realistic estimate of the generalization capability. Let's consider  four main metrics:
- Precision: share of positive predictions that are truly positive;
- Recall: share of positive instances correctly retrieved;
- F1-score: harmonic mean of precision and recall; 
- Accuracy: share of correct predictions out of the total.

In [ ]:
best_est = rs_best.best_estimator_    # Get the best pipeline from the refinement phase 
y_pred = best_est.predict(X_test)     # Predict on the test set

f1_score  = f1_score(y_test, y_pred)    # F1 = 2 * (precision*recall) / (precision+recall)
precision = precision_score(y_test, y_pred)    # TP / (TP+FP)
recall = recall_score(y_test, y_pred)    # TP / (TP+FN)
accuracy  = accuracy_score(y_test, y_pred)    # (TP+TN) / (TP+FP+FN+TN)

print("F1 Score:", f1_score)
print("Precision:", precision)
print("Recall:", recall)
print("Accuracy:", accuracy)

### Confusion matrix 

The confusion matrix is visualized to analyze how precision, recall, F1 score, and accuracy are derived. It breaks down predictions into True Positives (TP), False Positives (FP), True Negatives (TN), and False Negatives (FN).

In [ ]:
cm = confusion_matrix(y_test, y_pred)    # 2x2 matrix 

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Introvert(0)", "Extrovert (1)"])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix', fontsize=14)
plt.xlabel('Predicted Labels', fontsize=12)
plt.ylabel('True Labels', fontsize=12)
plt.tight_layout()
plt.show()

The confusion matrix reveals the following:
- false positives are few (18), hence the high precision: most of the predicted positives are actually positive;
- false negatives are slightly more (26) than false positives, hence the slightly lower recall: some real positives are still missed;
- the overall correctness (TP+TN) on 580 examples is 536, leading to an accuracy of ≈ 92.4%.
Overall, the model generalizes well on the original test set: oversampling and hyperparameter optimization have led to a balanced classifier, with a slight bias toward reducing false positive errors.

### Precision-Recall Curve (PRC)

The precision–recall curve is useful to observe how precision and recall change as the decision threshold varies.

In [ ]:
best_est = rs_best.best_estimator_   # pipeline selected in the refinement phase 

if hasattr(best_est, "predict_proba"):
    y_scores = best_est.predict_proba(X_test)[:, 1]
else:
    y_scores = best_est.decision_function(X_test)

precision, recall, thresholds = precision_recall_curve(y_test, y_scores)

# Plot the curve
plt.figure(figsize=(6,5))
plt.plot(recall, precision, lw=2, label="Precision-Recall")
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

The curve indicates a strong but not perfect balance between precision and recall. 
- With lower thresholds, recall increases toward 1.0, but precision drops to 0.55-0.60 due to the increase in false positives.
- With higher thresholds, precision remains very high (0.93-0.95), but recall decreases.
- A good compromise is observed around 0.85-0.90 recall with 0.94 precision.
Overall, the curve is high for most of the range, indicating good performance, although it highlights the trade-off: it is not possible to maintain very high precision when pushing recall toward 1.0.

### ROC curve

The Receiver Operating Characteristic (ROC) curve evaluates the classifier's behavior as the decision threshold varies. For each threshold, the following are calculated:
- TPR/Recall (y-axis): percentage of positives correctly identified
- FPR (x-axis): percentage of negatives mistaken for positives.
The ROC curve shows the tradeoff between TPR and FPR for all thresholds; the area under the curve  (AUC) summarizes the model's ranking performance.

In [ ]:
y_score = best_est.decision_function(X_test)
fpr, tpr, thresholds = roc_curve(y_test, y_score)    # Compute ROC curve

roc_auc = auc(fpr, tpr)    # Compute the AUC
print(f"AUC (ROC): {roc_auc:.3f}")

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random (AUC = 0.5)")  # riferimento casuale
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(alpha=0.2)
plt.show()

The resulting ROC curve shows a marked trend toward the upper left corner, clearly above the random diagonal. The AUC = 0.923 indicates excellent discrimination in most cases. In the initial region, the curve rises very rapidly, indicating that it is possible to achieve TPR of 0.90–0.93 while maintaining an FPR < 0.10; beyond this threshold, more significant increases in false positives are required to increase recall. Overall, the result is consistent with the obtained F1, precision, and recall and confirms good generalization.

### Learning curve

The learning curve shows how the quality of the model changes as the amount of training data varies. This analysis helps identify issues such as overfitting or underfitting by comparing the model's accuracy on the training and validation sets.

In [ ]:
cls = rs_best.best_estimator_     # Final model to evaluate 
cv = StratifiedKFold(n_splits=5, shuffle=False)

train_fracs = [0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]    # Percentuali di training set da provare (come nell'esempio)

# Calcolo delle curve di apprendimento con F1 come metrica
train_sizes, train_scores, test_scores = learning_curve(
    estimator=cls,
    X=X_train,
    y=y_train,
    train_sizes=train_fracs,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    shuffle=False
)

train_mean = np.mean(train_scores, axis=1)
train_std  = np.std(train_scores, axis=1)
test_mean  = np.mean(test_scores, axis=1)
test_std   = np.std(test_scores, axis=1)


fig = plt.figure(figsize=(12, 7))
ax = fig.add_subplot(1, 1, 1)
ax.plot(train_sizes, train_mean,
        color='blue', marker='+', markersize=5, label='Training F1')
ax.fill_between(train_sizes,
                train_mean + train_std,
                train_mean - train_std,
                alpha=0.15, color='blue')
ax.plot(train_sizes, test_mean,
        color='green', linestyle='--', marker='d', markersize=5, label='Validation F1')
ax.fill_between(train_sizes,
                test_mean + test_std,
                test_mean - test_std,
                alpha=0.15, color='green')
ax.grid(True)
ax.set_xlabel('Training set size')
ax.set_ylabel('F1-score')
ax.legend(loc='lower right')
ax.set_ylim([0.60, 1.03])
plt.show()

After the initial phase which is very noisy with few examples, the training curve quickly stabilizes around 0.95, while the validation curve remains stable around 0.93–0.94. The two curves are close and almost parallel along the entire axis: the gap of 0.01–0.02 indicates no significant overfitting and good generalization ability. The error bands narrow as the data grow, a sign that the performance estimate becomes more stable.
In summary, the model is well-tuned: high F1, small gap, and low variance.

### Validation curve

The validation curve is visualized to analyze the model's performance as a function of a specific hyperparameter, in order to understand where under- and over-regularization occurs. In this case, SVC is analyzed within the selected pipeline, varying the C parameter on a logarithmic scale. [[[C controls the intensity of the regularization: small values ​​impose strict margins (a simple model), large values ​​allow for a better fit to the data.]]]

In [ ]:
range_C = [0.001,0.01,0.1,1,10,100]
train_scores, test_scores = validation_curve(cls,
        X=X_train, 
        y=y_train, 
        param_range=
        range_C, 
        param_name='classifier__C',
        cv=5, 
        n_jobs=-1, 
        scoring='f1'
)

In [ ]:
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

fig=plt.figure(figsize=(12,7))
ax = fig.add_subplot()
ax.plot(range_C, train_mean,
         color='blue', marker='o',
         markersize=5, label='Training F1')
ax.fill_between(range_C,
                 train_mean + train_std,
                 train_mean - train_std,
                 alpha=0.15, color='blue')
ax.plot(range_C, test_mean,
         color='green', linestyle='--',
         marker='s', markersize=5,
         label='Validation F1')
ax.fill_between(range_C,
                 test_mean + test_std,
                 test_mean - test_std,
                 alpha=0.15, color='green')

ax.grid()
ax.set_xlabel('Parameter C')
ax.set_ylabel('F1-score')
ax.legend(loc='lower right')
ax.set_xscale('log')

# Zoom on y axis
ymin = min(train_mean.min(), test_mean.min()) - 0.0008   
ymax = max(train_mean.max(), test_mean.max()) + 0.0008
ax.set_ylim(ymin, ymax)    

The validation curve shows that the average F1 for training and validation remains practically flat across the entire range. The two curves almost coincide: the deviation is on the order of a few thousandths of the F1, as shown by the zoom on the y-axis. This indicates that, in the explored range, the model's performance is not sensitive to the choice of C; neither overfitting nor underfitting are observed. In practice, any value of C in this range produces equivalent results: it is therefore possible to prefer a simple robust value without expecting significant changes in performance.

# Conclusion 

The project's goal was to build a binary classifier capable of predicting whether a person is Extroverted (1) or Introverted (0) starting from behavioral indicators and social habits. The entire process was implemented in a single pipeline: missing value imputation, standardization/encoding via ColumnTransformer, imbalance management with RandomOverSampler, and the final SVC model. Model selection was conducted with nested cross-validation: the chosen candidate was then refined with a further RandomizedSearchCV, varying the most influential parameters and the oversampling sampling strategy.

On the test data, the final model achieved solid and balanced performance: F1 = 0.925, precision = 0.938, recall = 0.913, and accuracy = 0.924. The ROC curve shows an area under the curve of AUC = 0.923, indicating good separation between classes; the Precision–Recall curve confirms that precision remains high until recall is pushed to extreme values, where a decline is natural due to the increase in false positives. The learning curves show very close and stable training and validation trajectories, indicating low variance and appropriate overfitting control; the validation curve for the
 C parameter is almost flat, confirming the model's robustness to reasonable variations in the hyperparameters.

Overall, the approach adopted led to a reliable classifier, which generalizes well and is insensitive to small parameter perturbations. Compared to the initial objectives, the delivered system is consistent, stable, and ready for use.